# 🎓 Class Twin — Phase 1
### An AI twin that listens to a class question and answers like me, in Egyptian Arabic or English

> ⚠️ **Responsible use:** run it only in meetings where the instructor knows it is running. Never use it for attendance, quizzes/exams, or to attend a session in my place. Clone only my own voice and keep `voice/` out of public repos.

---
## 🗺️ How it works

```
 🎤 audio ──► 1. TRANSCRIBE ──► 2. DECIDE ──► 3. REPLY ──► 4. SPEAK ──► 🔊 mp3
              Groq Whisper      decision tree   dependency     edge-tts
              text+quality      + pruning       graph + LLM
```

| Step | File | What it does |
|---|---|---|
| 1 | `stt.py` | Speech → text, detected language, audio-quality score |
| 2 | `decide.py` | Chooses: **answer / ask_to_repeat / defer / stay_silent** |
| 3 | `planner.py` | Runs the tools in parallel, then the LLM writes the reply in my style |
| 4 | `speak.py` | Text → voice file |

---
## ✅ Before you start
1. Your `twin_data/` folder (`profile.json`, `style_examples.jsonl`, `knowledge/`, `voice/`) must be inside `PROJECT_DIR` on Google Drive.
2. Add `GROQ_API_KEY` in **Colab Secrets** (the 🔑 icon on the left).
3. Run the cells **from top to bottom**.

---
# ⚙️ Part 1 — Setup
## 1.1 Install the libraries
Installs Groq (speech-to-text + LLM), LangChain tools, sentence-transformers + FAISS (search), edge-tts (voice), and Gradio (demo UI).

In [ ]:
!pip install -q groq langchain-core langchain-groq sentence-transformers faiss-cpu edge-tts gradio "pydantic>=2"

## 1.2 Connect Google Drive, set paths and API key
Mounts Drive, moves into the project folder, and tells the code where `twin_data/` is. If your folder has another name, change `PROJECT_DIR`.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/ClassTwin"
os.makedirs(PROJECT_DIR + "/tests/audio", exist_ok=True)
%cd {PROJECT_DIR}

os.environ["TWIN_DATA"] = PROJECT_DIR + "/twin_data"   # profile.json, style_examples.jsonl, knowledge/
os.environ["TWIN_OUT"] = PROJECT_DIR + "/outputs"

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

assert os.path.exists(os.environ["TWIN_DATA"] + "/profile.json"), "twin_data/profile.json not found - check PROJECT_DIR"
print("✅ ready")

In [ ]:
from pathlib import Path
import subprocess, sys

# --- find the project folder in Drive ---
BASE = Path("/content/drive/MyDrive/ClassTwin")
if not (BASE / "textutils.py").exists():
    hits = list(Path("/content/drive").glob("**/ClassTwin/textutils.py"))
    assert hits, "ClassTwin folder not found - mount Drive first"
    BASE = hits[0].parent
print("Project folder:", BASE)

def patch(fname, old, new, marker):
    p = BASE / fname
    s = p.read_text(encoding="utf-8")
    if marker in s:
        print(f"{fname}: already patched, skipped")
        return
    if old not in s:
        print(f"{fname}: !! could not find the target text, nothing changed")
        return
    p.write_text(s.replace(old, new, 1), encoding="utf-8")
    print(f"{fname}: patched")

# 1) Arabic comma bug (textutils.py)
# The range \u0600-\u06FF also contains Arabic punctuation like "،" and "؟",
# so "كريم،" was read as one word. Letters only: \u0621-\u064A.
p = BASE / "textutils.py"
s = p.read_text(encoding="utf-8")
if "\\u0600-\\u06FF" in s:
    p.write_text(s.replace("\\u0600-\\u06FF", "\\u0621-\\u064A"), encoding="utf-8")
    print("textutils.py: Arabic comma fix applied")
else:
    print("textutils.py: already patched, skipped")

# 2) Reply prompt: no invented stories (planner.py)
patch(
    "planner.py",
    "Never invent numbers, results or experience.",
    "Never invent numbers, results or experience. "
    "NO PERSONAL STORIES: do not say things like 'I tried', 'in my project I noticed' or 'once I saw' "
    "unless that exact experience is written in NOTES. Explain the concept in general terms instead. "
    "If asked about an experience that is not in NOTES, say honestly that I have not worked on it yet.",
    marker="NO PERSONAL STORIES",
)

# 3) Decision prompt: stricter scoring (decide.py)
patch(
    "decide.py",
    "Be strict: a topic that only looks similar is NOT covered.",
    "Be strict: a topic that only looks similar is NOT covered. "
    "If a good answer would need a personal story or experience that is not written in my notes, "
    "the answer score must be below 0.5.",
    marker="personal story or experience that is not written",
)

# 4) Quick self-test in a fresh process
test = '''
from textutils import find_vocative, name_match, words
print(words("كريم، إيه الفرق؟"))
v = find_vocative("يا كريم، إيه الفرق بين RAG و fine-tuning؟")
print("vocative:", repr(v), "-> is me:", bool(name_match(v)))
v = find_vocative("يا سارة، شاركي الشاشة")
print("vocative:", repr(v), "-> is me:", bool(name_match(v)))
'''
r = subprocess.run([sys.executable, "-c", test], cwd=BASE, capture_output=True, text=True)
print(r.stdout or r.stderr)

---
# 🧱 Part 2 — The code
Each cell below writes **one file** into the project folder. Run them all once.

## 2.1 `config.py` — settings
All the numbers you may want to tune live here: models, the audio-quality limit, the decision thresholds, the voices, and the memory size.

In [ ]:
%%writefile config.py
"""Central settings. Everything you may want to tune lives here."""
import json
import os
from functools import lru_cache
from pathlib import Path

ROOT = Path(__file__).resolve().parent
DATA_DIR = Path(os.getenv("TWIN_DATA", ROOT / "twin_data"))
OUT_DIR = Path(os.getenv("TWIN_OUT", ROOT / "outputs"))
(OUT_DIR / "audio").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tests").mkdir(parents=True, exist_ok=True)

# models
STT_MODEL = "whisper-large-v3-turbo"
LLM_MODEL = "llama-3.3-70b-versatile"
EMBED_MODEL = "intfloat/multilingual-e5-small"
# hints Whisper about names and terms (helps spelling of Karim / كريم and technical words)
WHISPER_PROMPT = "كريم, Karim, RAG, fine-tuning, embedding, decision tree, LLM, FAISS, AgriVision"

# decision policy
QUALITY_MIN = 0.45          # below this the audio is "unclear" -> ask_to_repeat
DECISION_THRESHOLD = 0.60   # a branch must score at least this to survive pruning
OPEN_Q_ANSWER_MIN = 0.80    # stricter bar for answering an open question to the class

# language detection (script based, see textutils.detect_language)
MIXED_MIN_LATIN_WORDS = 3
MIXED_MIN_LATIN_SHARE = 0.25
NAME_MATCH_RATIO = 0.80     # fuzzy match for my name in the transcript

MEMORY_TURNS = 8

# edge-tts voices (Karim is male, so Shakir for Arabic)
VOICES = {
    "ar": "ar-EG-ShakirNeural",
    "mixed": "ar-EG-ShakirNeural",
    "en": "en-US-AndrewNeural",
}


@lru_cache(maxsize=1)
def load_profile() -> dict:
    return json.loads((DATA_DIR / "profile.json").read_text(encoding="utf-8"))


Writing config.py


## 2.2 `schemas.py` — data models
Defines `TwinReply` (the final structured answer: addressed_to_me, action, language, reply_text, confidence, tools_used, sources) and `Branch` (one option in the decision tree).

In [ ]:
%%writefile schemas.py
from typing import Literal, Optional
from pydantic import BaseModel, Field

Action = Literal["answer", "ask_to_repeat", "defer", "stay_silent"]
Lang = Literal["ar", "en", "mixed"]


class TwinReply(BaseModel):
    addressed_to_me: bool
    action: Action
    language: Lang
    reply_text: str
    confidence: float = Field(ge=0, le=1)
    tools_used: list[str] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)


class Branch(BaseModel):
    """One candidate action in the decision tree."""
    action: Action
    kind: Literal["check", "llm", "check+llm"]
    threshold: float
    score: Optional[float] = None
    status: Literal["pending", "kept", "pruned", "selected"] = "pending"
    reason: str = ""


Writing schemas.py


## 2.3 `textutils.py` — cheap text checks (no LLM)
Fast rules used before any LLM call: detect Arabic / English / **mixed**, fuzzy-match my name (Karim / Kareem / كريم), spot who is being called (`Sara, ...`), detect open questions to the class, and follow-up questions.

In [ ]:
%%writefile textutils.py
"""Cheap text checks (no LLM): language, my name, vocatives, questions."""
import difflib
import re
from functools import lru_cache

import config

_DIAC = re.compile("[\u064B-\u0652\u0640]")           # tashkeel + tatweel
_WORD = re.compile(r"[\w\u0600-\u06FF]+(?:-\w+)*")     # keeps fine-tuning as one word
_AR = re.compile(r"[\u0600-\u06FF]")
_LAT = re.compile(r"[a-z]")

_NOT_NAMES = {"okay", "ok", "so", "well", "now", "alright", "right", "yes", "no", "hello",
              "hi", "hey", "also", "and", "but", "actually", "anyway", "first", "next",
              "today", "everyone", "anyone", "guys", "class", "جماعه", "شباب", "بنات",
              "دكتور", "استاذ", "ريس"}

_OPEN = ["anyone", "anybody", "any one", "everyone", "everybody", "who can", "who knows",
         "who wants", "guys", "حد ", "اي حد", "حد منكم", "حد يعرف", "يا جماعه", "يا شباب",
         "مين يعرف", "مين يقدر"]

_Q_EN = re.compile(r"\b(what|why|how|when|where|which|who|can you|could you|would you|do you|"
                   r"does|is it|are you|explain|tell us|difference)\b", re.I)
_Q_AR = {"ايه", "ليه", "ازاي", "ممكن", "هل", "مين", "فين", "امتي", "كام", "اشرح", "فرق", "الفرق"}

_FOLLOW_EN = re.compile(r"\b(that|earlier|before|you said|previous|mentioned|as well)\b", re.I)
_FOLLOW_AR = ["قبل كده", "قلته", "قلت", "اللي فات", "اللي قلته"]


def norm(text: str) -> str:
    t = _DIAC.sub("", text.lower())
    t = re.sub("[إأآ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    return t


def words(text: str) -> list[str]:
    return _WORD.findall(norm(text))


@lru_cache(maxsize=1)
def name_tokens() -> frozenset:
    p = config.load_profile()
    toks = set()
    for n in p.get("name_spellings", []) + [p.get("first_name", "")]:
        for w in words(n):
            if len(w) >= 3:
                toks.add(w)
    return frozenset(toks)


def name_match(text: str):
    """Return the transcript word that matches my name (fuzzy), else None."""
    for w in words(text):
        if len(w) < 3:
            continue
        for n in name_tokens():
            if w == n or difflib.SequenceMatcher(None, w, n).ratio() >= config.NAME_MATCH_RATIO:
                return w
    return None


def detect_language(text: str) -> str:
    """'ar' / 'en' / 'mixed' from the script of the words (more reliable than Whisper's label
    for code-switched Egyptian Arabic). My own name is ignored so 'Karim' in an Arabic sentence
    does not count as English."""
    names = name_tokens()
    ws = [w for w in words(text) if w not in names]
    ar = [w for w in ws if _AR.search(w)]
    la = [w for w in ws if _LAT.search(w)]
    if not ar:
        return "en"
    if not la:
        return "ar"
    share = len(la) / len(ws)
    if len(la) >= config.MIXED_MIN_LATIN_WORDS or share >= config.MIXED_MIN_LATIN_SHARE:
        return "mixed"
    return "ar"


def find_vocative(text: str):
    """The person being called at the start of the sentence ('Sara, ...' / 'يا سارة ...')."""
    t = text.strip()
    m = re.match(r"^(?:(?:hey|hi|hello|ok|okay)\s+)?([A-Za-z]{2,})\s*[,،:]", t, re.I)
    v = m.group(1) if m else None
    if not v:
        m = re.match(r"^\s*يا\s+([\u0600-\u06FF]+)", t)
        v = m.group(1) if m else None
    if v and norm(v) in _NOT_NAMES:
        return None
    return v


def is_open_question(text: str) -> bool:
    t = norm(text) + " "
    return any(m in t for m in _OPEN)


def is_question(text: str) -> bool:
    if "?" in text or "؟" in text or _Q_EN.search(text):
        return True
    ws = set(words(text))
    return bool(ws & _Q_AR) or "يعني ايه" in norm(text)


def is_followup(text: str) -> bool:
    t = norm(text)
    return bool(_FOLLOW_EN.search(text)) or any(m in t for m in _FOLLOW_AR)


def make_query(question: str, last_said: str) -> str:
    """Follow-ups ('how is that different from what you said earlier?') carry no topic,
    so we add what I said last to the search query."""
    if last_said and is_followup(question):
        return f"{question} {last_said[:200]}"
    return question


Writing textutils.py


## 2.4 `memory.py` — meeting memory
Keeps a rolling transcript of what the class said **and** what the twin already answered, so follow-up questions like *"what you said earlier"* work.

In [ ]:
%%writefile memory.py
"""Rolling meeting memory: what was heard + what the twin already said."""
from config import MEMORY_TURNS


class Memory:
    def __init__(self, max_turns: int = MEMORY_TURNS):
        self.max_turns = max_turns
        self.reset()

    def reset(self):
        self.turns: list[tuple[str, str]] = []   # ("heard" | "me", text)
        self.last_speaker = None

    def add_heard(self, text: str):
        if text.strip():
            self._add("heard", text.strip())

    def add_said(self, text: str):
        if text.strip():
            self._add("me", text.strip())

    def _add(self, who, text):
        self.turns.append((who, text))
        self.turns = self.turns[-self.max_turns * 2:]
        self.last_speaker = who

    def last_said(self) -> str:
        for who, t in reversed(self.turns):
            if who == "me":
                return t
        return ""

    def context(self) -> str:
        if not self.turns:
            return "(nothing yet)"
        return "\n".join(f"{'ME (twin)' if w == 'me' else 'CLASS'}: {t}" for w, t in self.turns)


Writing memory.py


## 2.5 `retrieval.py` — search engine
Embeds my notes and my Q&A examples with `multilingual-e5-small` (Arabic + English in one index) and searches them with FAISS. Adds the `query:` / `passage:` prefixes the model needs.

In [ ]:
%%writefile retrieval.py
"""FAISS + multilingual-e5-small. Heavy imports are lazy so the other modules load fast."""
import json
import threading
from functools import lru_cache

import config

_lock = threading.Lock()


@lru_cache(maxsize=1)
def _embedder():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(config.EMBED_MODEL)


def _encode(texts):
    with _lock:
        return _embedder().encode(texts, normalize_embeddings=True,
                                  convert_to_numpy=True, batch_size=32).astype("float32")


@lru_cache(maxsize=256)
def _encode_query(q: str):
    return _encode(["query: " + q])


class Index:
    """doc_prefix: 'passage: ' for notes. For question-to-question matching e5 recommends
    'query: ' on both sides (symmetric task)."""

    def __init__(self, items, text_key="text", doc_prefix="passage: "):
        import faiss
        self.items = items
        embs = _encode([doc_prefix + it[text_key] for it in items])
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)

    def search(self, query: str, k: int = 3):
        k = min(k, len(self.items))
        scores, ids = self.index.search(_encode_query(query), k)
        return [(self.items[i], float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]


def chunk_text(text: str, max_chars: int = 700):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if cur and len(cur) + len(p) > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = f"{cur}\n\n{p}" if cur else p
    if cur:
        chunks.append(cur)
    return chunks


@lru_cache(maxsize=1)
def notes_index() -> Index:
    items = []
    for p in sorted((config.DATA_DIR / "knowledge").glob("**/*")):
        if p.suffix.lower() not in {".md", ".txt"}:
            continue
        text = p.read_text(encoding="utf-8")
        title = text.strip().splitlines()[0].lstrip("# ").strip() if text.strip() else p.stem
        for c in chunk_text(text):
            body = c if c.lstrip().startswith("#") else f"{title}\n{c}"
            items.append({"source": p.name, "text": body})
    if not items:
        raise RuntimeError(f"No notes found in {config.DATA_DIR / 'knowledge'}")
    return Index(items)


@lru_cache(maxsize=1)
def style_index() -> Index:
    path = config.DATA_DIR / "style_examples.jsonl"
    items = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    return Index(items, text_key="question", doc_prefix="query: ")


Writing retrieval.py


## 2.6 `tools.py` — the 4 `@tool` functions
| Tool | Job |
|---|---|
| `search_my_notes` | finds the most relevant chunks in my notes |
| `get_style_examples` | finds my past answers similar to the question (to copy my tone) |
| `get_profile` | returns my profile and rules |
| `lookup_glossary` | short definitions from my course glossary |

`call_tool()` runs a tool and logs its name, JSON arguments, and time.

In [ ]:
%%writefile tools.py
"""The four @tool functions + call_tool(), which runs a tool and logs name, JSON args and time."""
import json
import re
import threading
import time
from difflib import get_close_matches
from functools import lru_cache

from langchain_core.tools import tool

import config
import textutils as tu


@tool
def search_my_notes(query: str, k: int = 3) -> list[dict]:
    """Search my own notes (project write-ups, course notes) and return the k most relevant chunks."""
    from retrieval import notes_index
    return [{"source": it["source"], "text": it["text"], "score": round(s, 3)}
            for it, s in notes_index().search(query, k)]


@tool
def get_style_examples(question: str, language: str = "en", k: int = 4) -> list[dict]:
    """Return k of my past Q&A pairs most similar to the question, preferring the same language
    ('ar', 'en' or 'mixed'). Used to copy my tone, not my facts."""
    from retrieval import style_index
    pref = {"ar": ["ar", "mixed"], "mixed": ["mixed", "ar"], "en": ["en"]}.get(language, ["en"])
    hits = style_index().search(question, 12)
    ordered = [h for h in hits if h[0]["language"] in pref] + [h for h in hits if h[0]["language"] not in pref]
    return [{"question": it["question"], "answer": it["answer"],
             "language": it["language"], "score": round(s, 3)} for it, s in ordered[:k]]


@tool
def get_profile() -> dict:
    """Return my profile: name, background, how I speak, and the rules I follow."""
    return config.load_profile()


@lru_cache(maxsize=1)
def _glossary() -> dict:
    g = {}
    path = config.DATA_DIR / "knowledge" / "ml_glossary.md"
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            m = re.match(r"^- ([^:]+):\s*(.+)$", line.strip())
            if m:
                g[m.group(1).strip().lower()] = m.group(2).strip()
    return g


@tool
def lookup_glossary(term: str) -> dict:
    """Look up a short definition of an ML/GenAI term in my own course glossary."""
    g = _glossary()
    t = term.lower().strip()
    key = t if t in g else None
    if not key:   # token overlap, e.g. 'pruning decision tree' -> 'decision tree pruning'
        tt = set(tu.words(t))
        best, best_s = None, 0.5
        for k in g:
            kt = set(tu.words(k))
            s = len(tt & kt) / max(len(kt), 1)
            if s > best_s:
                best, best_s = k, s
        key = best
    if not key:
        close = get_close_matches(t, list(g), n=1, cutoff=0.6)
        key = close[0] if close else None
    return {"found": True, "term": key, "definition": g[key]} if key else {"found": False, "term": term}


_TOOLS = {t.name: t for t in [search_my_notes, get_style_examples, get_profile, lookup_glossary]}
CALL_LOG: list[dict] = []
_log_lock = threading.Lock()


def clear_call_log():
    with _log_lock:
        CALL_LOG.clear()


def call_tool(name: str, **args):
    t0 = time.perf_counter()
    out = _TOOLS[name].invoke(args)
    dt = time.perf_counter() - t0
    with _log_lock:
        CALL_LOG.append({"tool": name, "args": args, "seconds": round(dt, 3),
                         "result_preview": json.dumps(out, ensure_ascii=False)[:300]})
    return out


Writing tools.py


## 2.7 `stt.py` — speech to text
`transcribe(audio_path)` sends audio to Groq Whisper and returns the **text**, the **language**, and a **quality score** (from the segment confidence and no-speech probability). Low quality → the twin asks to repeat.

In [ ]:
%%writefile stt.py
"""transcribe(audio_path) -> text, language, quality score (Groq Whisper, verbose_json)."""
import math
import os

import config
import textutils as tu


def _get(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    v = getattr(obj, key, None)
    if v is None and getattr(obj, "model_extra", None):
        v = obj.model_extra.get(key)
    return default if v is None else v


def quality_score(segments, text: str) -> float:
    """0..1. exp(avg_logprob) is a per-segment confidence; (1 - no_speech_prob) penalises
    segments that are probably silence/noise. Segments are weighted by duration."""
    if not text.strip():
        return 0.0
    if not segments:
        return 0.5
    num = den = 0.0
    for s in segments:
        dur = max(float(_get(s, "end", 0)) - float(_get(s, "start", 0)), 0.05)
        conf = min(1.0, math.exp(float(_get(s, "avg_logprob", -1.0))))
        speech = 1.0 - float(_get(s, "no_speech_prob", 0.0))
        num += conf * speech * dur
        den += dur
    return round(num / den, 3)


def transcribe(audio_path: str) -> dict:
    from groq import Groq
    client = Groq()   # reads GROQ_API_KEY
    with open(audio_path, "rb") as f:
        resp = client.audio.transcriptions.create(
            file=(os.path.basename(audio_path), f.read()),
            model=config.STT_MODEL,
            response_format="verbose_json",
            temperature=0.0,
            prompt=config.WHISPER_PROMPT,
        )
    text = (_get(resp, "text", "") or "").strip()
    segs = _get(resp, "segments", []) or []
    return {
        "text": text,
        "language": tu.detect_language(text),
        "whisper_language": _get(resp, "language"),
        "quality": quality_score(segs, text),
        "duration": _get(resp, "duration"),
        "segments": [{"start": _get(s, "start"), "end": _get(s, "end"),
                      "avg_logprob": _get(s, "avg_logprob"),
                      "no_speech_prob": _get(s, "no_speech_prob")} for s in segs],
    }


def transcript_from_text(text: str, quality: float | None = None) -> dict:
    """Dev mode: skip speech-to-text and type the question."""
    return {"text": text.strip(), "language": tu.detect_language(text), "whisper_language": None,
            "quality": 0.9 if quality is None else quality, "duration": None, "segments": []}


Writing stt.py


## 2.8 `llm.py` — LLM helper
Small wrapper around Groq's Llama 3.3 70B that always returns clean JSON.

In [ ]:
%%writefile llm.py
import json
import re

from functools import lru_cache

import config


@lru_cache(maxsize=4)
def get_llm(temperature: float = 0.0):
    from langchain_groq import ChatGroq
    return ChatGroq(model=config.LLM_MODEL, temperature=temperature,
                    model_kwargs={"response_format": {"type": "json_object"}})


def llm_json(system: str, user: str, temperature: float = 0.0):
    """Call the LLM in JSON mode. Returns a dict, or None if the output is not valid JSON."""
    msg = get_llm(temperature).invoke([("system", system), ("user", user)])
    txt = msg.content if isinstance(msg.content, str) else str(msg.content)
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", txt, re.S)
        try:
            return json.loads(m.group(0)) if m else None
        except json.JSONDecodeError:
            return None


Writing llm.py


## 2.9 `decide.py` — the decision tree 🌳
Chooses what the twin does: **answer / ask_to_repeat / defer / stay_silent**.
1. **Cheap checks first** — audio quality, my name, who is being called, is it a question.
2. **The LLM scores only what is left.**
3. `prune_and_select()` removes branches under their threshold and picks the best one.
4. If nothing clears the threshold → a safe reply, never a guess.

Open questions to the class need a higher score (0.80) before the twin speaks.

In [ ]:
%%writefile decide.py
"""Response policy as a decision tree.

root: should I speak?
 |- ask_to_repeat  (cheap check: audio quality / too short)
 |- stay_silent    (cheap check: called someone else, or not a question)  + LLM score
 |- defer          (LLM score: it is for me but outside my notes)
 `- answer         (LLM score: it is for me and my notes/memory cover it)

Cheap checks run first. If one fires strongly, the LLM is never called and the LLM branches
are pruned. Otherwise the LLM scores only what is left. prune_and_select() drops every branch
below its threshold and picks the best survivor. If nothing survives -> safe reply.
"""
import config
import textutils as tu
from llm import llm_json
from schemas import Branch
from tools import call_tool

PRIORITY = ["ask_to_repeat", "stay_silent", "defer", "answer"]   # tie-break: safest first
CHEAP_FIRE = 0.9


def prune_and_select(branches: list[Branch]):
    for b in branches:
        b.status = "kept" if (b.score is not None and b.score >= b.threshold) else "pruned"
    kept = [b for b in branches if b.status == "kept"]
    if not kept:
        return None
    best = max(kept, key=lambda b: (b.score, -PRIORITY.index(b.action)))
    best.status = "selected"
    return best


def _features(tr: dict, memory) -> dict:
    text = tr["text"]
    name = tu.name_match(text)
    voc = tu.find_vocative(text)
    other = bool(voc) and tu.name_match(voc) is None
    q = tu.is_question(text)
    fu = tu.is_followup(text)
    continuing = fu and memory.last_speaker == "me"
    addressed = bool(name) or continuing
    open_q = (not addressed) and (not other) and (tu.is_open_question(text) or q)
    return {"text": text, "quality": tr["quality"], "language": tr["language"],
            "word_count": len(tu.words(text)), "name_in_text": name, "vocative": voc,
            "addressed_to_other": other, "is_question": q, "is_followup": fu,
            "addressed_to_me": addressed, "open_question": open_q}


def _llm_scores(f: dict, memory) -> tuple[dict, list]:
    query = tu.make_query(f["text"], memory.last_said())
    hits = call_tool("search_my_notes", query=query, k=3)
    notes = "\n".join(f"[{h['source']}] {h['text'][:350]}" for h in hits) or "(none)"
    system = ("You are the decision module of Karim's AI twin in a university class meeting. "
              "Output JSON only, with keys: answer, defer, stay_silent (numbers 0 to 1) and reason (short).")
    user = f"""Transcript of what was just said: "{f['text']}"
Addressed to me (Karim): {f['addressed_to_me']}
Open question to the whole class: {f['open_question']}
Follow-up of something I said: {f['is_followup']}

MEETING MEMORY:
{memory.context()}

MY NOTES (top matches):
{notes}

Score each action from 0 to 1:
- answer: I can answer correctly using ONLY my notes, my memory of this meeting, or my profile.
- defer: it is for me, but the notes/memory do not cover it, or it asks about work I did not do. I must not guess.
- stay_silent: it is not for me, needs no reply, or it is an open question to the class that my notes do not clearly cover (I never speak unprompted without good reason).
Be strict: a topic that only looks similar is NOT covered."""
    out = llm_json(system, user) or {}

    def num(k):
        try:
            return max(0.0, min(1.0, float(out.get(k, 0.0))))
        except (TypeError, ValueError):
            return 0.0
    return {"answer": num("answer"), "defer": num("defer"), "stay_silent": num("stay_silent"),
            "reason": str(out.get("reason", ""))}, hits


def decide(tr: dict, memory) -> dict:
    f = _features(tr, memory)
    th = config.DECISION_THRESHOLD
    ans_th = config.OPEN_Q_ANSWER_MIN if f["open_question"] else th
    B = {a: Branch(action=a, kind=k, threshold=(ans_th if a == "answer" else th))
         for a, k in [("ask_to_repeat", "check"), ("stay_silent", "check+llm"),
                      ("defer", "llm"), ("answer", "llm")]}

    # 1) cheap checks
    bad_audio = f["quality"] < config.QUALITY_MIN or f["word_count"] < 2
    B["ask_to_repeat"].score = 0.95 if bad_audio else 0.0
    B["ask_to_repeat"].reason = (f"quality {f['quality']:.2f} (min {config.QUALITY_MIN}), "
                                 f"{f['word_count']} words")
    cheap_silent = 0.0
    if f["addressed_to_other"]:
        cheap_silent, why = 0.9, f"called someone else: '{f['vocative']}'"
    elif not f["addressed_to_me"] and not f["is_question"]:
        cheap_silent, why = 0.9, "no question and my name was not said"
    else:
        why = "no cheap reason to stay silent"
    B["stay_silent"].score = cheap_silent
    B["stay_silent"].reason = why

    # 2) LLM only for what is left
    hits, llm_reason = [], ""
    if max(B["ask_to_repeat"].score, cheap_silent) >= CHEAP_FIRE:
        for a in ("defer", "answer"):
            B[a].score = None
            B[a].reason = "not scored: a cheap check already decided"
    else:
        s, hits = _llm_scores(f, memory)
        llm_reason = s["reason"]
        B["answer"].score, B["defer"].score = s["answer"], s["defer"]
        B["stay_silent"].score = max(cheap_silent, s["stay_silent"])
        B["answer"].reason = f"LLM score (threshold {ans_th:.2f}" + (", open question)" if f["open_question"] else ")")
        B["defer"].reason = "LLM score"
        B["stay_silent"].reason += f"; LLM {s['stay_silent']:.2f}"

    # 3) prune and select
    branches = [B[a] for a in PRIORITY]
    chosen = prune_and_select(branches)
    fallback = chosen is None
    if fallback:
        # Safe default: if the question was for me, ask to repeat; if it was an open question
        # to the class, do not interrupt - stay silent.
        chosen = B["stay_silent"] if f["open_question"] else B["ask_to_repeat"]
        chosen.status = "selected"
        chosen.reason += " | FALLBACK: nothing cleared the threshold -> safe default, no guessing"

    return {"features": f, "selected": chosen.action, "fallback": fallback,
            "language": f["language"], "addressed_to_me": f["addressed_to_me"],
            "answer_score": B["answer"].score or 0.0, "llm_reason": llm_reason,
            "notes_preview": [{"source": h["source"], "score": h["score"]} for h in hits],
            "branches": [b.model_dump() for b in branches]}


def render_tree(d: dict) -> str:
    f = d["features"]
    mark = {"selected": "★ SELECTED", "kept": "✓ kept", "pruned": "✂ pruned", "pending": "?"}
    lines = [f"root: should I speak?   (threshold {config.DECISION_THRESHOLD})",
             f"  name heard: {f['name_in_text']} | vocative: {f['vocative']} | question: {f['is_question']} "
             f"| open: {f['open_question']} | follow-up: {f['is_followup']} | quality: {f['quality']}"]
    for i, b in enumerate(d["branches"]):
        last = i == len(d["branches"]) - 1
        score = "n/a" if b["score"] is None else f"{b['score']:.2f}"
        lines.append(f"{'└─' if last else '├─'} {b['action']:<13} [{b['kind']}] score {score} "
                     f"(min {b['threshold']:.2f})  {mark[b['status']]}")
        lines.append(f"{'   ' if last else '│  '}   why: {b['reason']}")
    if d["llm_reason"]:
        lines.append(f"LLM reason: {d['llm_reason']}")
    return "\n".join(lines)


Writing decide.py


## 2.10 `planner.py` — reply plan as a dependency graph
The reply is a graph run in `graphlib` order. The four tool nodes have no dependencies, so they run **in parallel**; then the prompt is built; then the LLM writes the answer in my voice and language.

In [ ]:
%%writefile planner.py
"""Reply = dependency graph run in graphlib order, with parallel branches."""
import time
from concurrent.futures import ThreadPoolExecutor
from graphlib import TopologicalSorter

import textutils as tu
from config import load_profile
from llm import llm_json
from schemas import TwinReply
from tools import CALL_LOG, call_tool

# node -> the nodes it depends on. The four tool nodes are independent (run in parallel).
GRAPH = {
    "get_profile": set(),
    "search_my_notes": set(),
    "get_style_examples": set(),
    "lookup_glossary": set(),
    "build_prompt": {"get_profile", "search_my_notes", "get_style_examples", "lookup_glossary"},
    "llm_reply": {"build_prompt"},
}

_STOP = {"what", "does", "do", "is", "the", "a", "an", "of", "in", "and", "to", "how", "why", "can",
         "you", "your", "me", "mean", "means", "difference", "between", "explain", "tell", "us",
         "that", "this", "about", "are", "it", "for", "karim"}


def run_graph(graph: dict, funcs: dict, max_workers: int = 4):
    """Run nodes in topological order; every wave of ready nodes runs in parallel.
    Raises graphlib.CycleError if the plan has a cycle."""
    ts = TopologicalSorter(graph)
    ts.prepare()
    results, waves, node_secs = {}, [], {}

    def timed(name):
        t0 = time.perf_counter()
        out = funcs[name](results)
        node_secs[name] = round(time.perf_counter() - t0, 3)
        return out

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        while ts.is_active():
            ready = list(ts.get_ready())
            waves.append(ready)
            futs = {n: ex.submit(timed, n) for n in ready}
            for n, fut in futs.items():
                results[n] = fut.result()
                ts.done(n)
    return results, waves, node_secs


def _glossary_term(question: str) -> str:
    latin = [w for w in tu.words(question) if w.isascii() and w not in _STOP and w not in tu.name_tokens()]
    return " ".join(latin) or question


def _lang_rule(lang: str) -> str:
    return {"ar": "Egyptian Arabic (عامية مصرية), NOT formal Arabic. Technical terms stay in English letters.",
            "mixed": "Egyptian Arabic mixed with English technical terms in Latin letters, the way I talk.",
            "en": "English."}[lang]


def _build_prompt(lang, question, memory, r):
    prof = r["get_profile"]
    style = prof["speaking_style"]
    notes = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in r["search_my_notes"]) or "(no notes found)"
    gl = r["lookup_glossary"]
    glossary = f"{gl['term']}: {gl['definition']}" if gl.get("found") else "(no glossary hit)"
    shots = "\n".join(f"Q: {e['question']}\nA: {e['answer']}" for e in r["get_style_examples"])
    system = f"""You are Karim's AI twin in a university class meeting. Answer exactly as Karim would: first person, {style['tone']}.
LANGUAGE: {_lang_rule(lang)}
LENGTH: {style['typical_answer_length']}.
STRUCTURE: {style['explains_with']}.
RULES:
- Use ONLY facts found in NOTES, GLOSSARY, MEETING MEMORY or PROFILE. If a detail is missing, say you are not sure. Never invent numbers, results or experience.
- STYLE EXAMPLES show my voice (tone, fillers, terms kept in English). Copy the voice, NOT their facts, unless NOTES support them.
- Keep these terms in English: {', '.join(style['keep_in_english'])}.
- {' '.join(prof['honesty_rules'])}
- Return JSON only: {{"reply_text": str, "confidence": number 0-1, "sources": [note file names you used]}}"""
    user = f"""QUESTION: {question}

MEETING MEMORY:
{memory.context()}

PROFILE BACKGROUND:
{chr(10).join('- ' + b for b in prof['background'])}

NOTES:
{notes}

GLOSSARY: {glossary}

STYLE EXAMPLES:
{shots}"""
    return system, user


def _canned(action: str, lang: str) -> str:
    safe = load_profile()["safe_replies"]
    if action == "stay_silent":
        return ""
    ar = lang in ("ar", "mixed")
    if action == "ask_to_repeat":
        return safe["ask_to_repeat_ar" if ar else "ask_to_repeat_en"]
    return safe["defer_ar" if ar else "defer_en"]


def reply(decision: dict, tr: dict, memory) -> tuple[TwinReply, dict]:
    action, lang = decision["selected"], decision["language"]
    plan = {"waves": [], "node_seconds": {}, "timings": {}}
    base = dict(addressed_to_me=decision["addressed_to_me"], action=action, language=lang)

    if action != "answer":
        return TwinReply(**base, reply_text=_canned(action, lang), confidence=1.0 if action != "defer" else 0.9,
                         tools_used=[c["tool"] for c in CALL_LOG], sources=[]), plan

    q = tr["text"]
    query = tu.make_query(q, memory.last_said())
    funcs = {
        "get_profile": lambda r: call_tool("get_profile"),
        "search_my_notes": lambda r: call_tool("search_my_notes", query=query, k=3),
        "get_style_examples": lambda r: call_tool("get_style_examples", question=q, language=lang, k=4),
        "lookup_glossary": lambda r: call_tool("lookup_glossary", term=_glossary_term(q)),
        "build_prompt": lambda r: _build_prompt(lang, q, memory, r),
        "llm_reply": lambda r: llm_json(*r["build_prompt"], temperature=0.3),
    }
    t0 = time.perf_counter()
    results, waves, node_secs = run_graph(GRAPH, funcs)
    plan["waves"], plan["node_seconds"] = waves, node_secs
    plan["timings"] = {"tools_parallel": round(max(node_secs[n] for n in waves[0]), 3),
                       "llm": node_secs["llm_reply"], "plan_total": round(time.perf_counter() - t0, 3)}

    out = results["llm_reply"] or {}
    text = str(out.get("reply_text", "")).strip()
    if not text:   # LLM failed -> never guess, fall back to the safe reply
        return TwinReply(**{**base, "action": "ask_to_repeat"}, reply_text=_canned("ask_to_repeat", lang),
                         confidence=0.0, tools_used=list(dict.fromkeys(c["tool"] for c in CALL_LOG)),
                         sources=[]), plan
    try:
        conf = max(0.0, min(1.0, float(out.get("confidence", 0.5))))
    except (TypeError, ValueError):
        conf = 0.5
    available = {h["source"] for h in results["search_my_notes"]}
    sources = [s for s in out.get("sources", []) if s in available] or sorted(available)[:1]
    return TwinReply(**base, reply_text=text, confidence=round(min(conf, decision["answer_score"] or conf), 2),
                     tools_used=list(dict.fromkeys(c["tool"] for c in CALL_LOG)), sources=sources), plan


Writing planner.py


## 2.11 `speak.py` — text to voice 🔊
`speak(reply)` saves an mp3 with edge-tts and picks the voice from `reply.language` (Egyptian Arabic male voice for `ar` / `mixed`, English male voice for `en`). Returns nothing when muted or when the action is `stay_silent`.

In [ ]:
%%writefile speak.py
import asyncio
import threading
import time

import config
from schemas import TwinReply


def _run(coro):
    """Works inside Colab/Jupyter too (where an event loop is already running)."""
    err = []

    def target():
        try:
            asyncio.run(coro)
        except Exception as e:   # noqa
            err.append(e)
    t = threading.Thread(target=target)
    t.start()
    t.join()
    if err:
        raise err[0]


def speak(reply: TwinReply, muted: bool = False):
    """Save the reply as an mp3 with edge-tts. Voice depends on reply.language.
    Returns the file path, or None when there is nothing to say (stay_silent) or the kill switch is on."""
    if muted or reply.action == "stay_silent" or not reply.reply_text.strip():
        return None
    import edge_tts
    path = config.OUT_DIR / "audio" / f"reply_{int(time.time() * 1000)}.mp3"
    _run(edge_tts.Communicate(reply.reply_text, config.VOICES[reply.language]).save(str(path)))
    return str(path)


Writing speak.py


## 2.12 `twin.py` — one full turn
`run_turn()` chains transcribe → decide → reply → speak, updates the memory, and logs the **seconds each stage takes** to `outputs/timings.csv`. Phase 2 will call these same functions inside a live loop.

In [ ]:
%%writefile twin.py
"""One full turn: transcribe -> decide -> reply -> speak, with per-stage timings.
Phase 2 can call these same four functions inside a live loop."""
import csv
import time
from contextlib import contextmanager

import config
from decide import decide, render_tree
from memory import Memory
from planner import reply
from speak import speak
from stt import transcribe, transcript_from_text
from tools import CALL_LOG, clear_call_log

MEMORY = Memory()
_turn = {"n": 0}


@contextmanager
def stage(timings: dict, name: str):
    t0 = time.perf_counter()
    yield
    timings[name] = round(time.perf_counter() - t0, 3)


def run_turn(audio_path=None, text=None, fake_quality=None, memory=MEMORY, muted=False) -> dict:
    _turn["n"] += 1
    timings = {}
    clear_call_log()
    t_all = time.perf_counter()

    with stage(timings, "speech_to_text"):
        tr = transcript_from_text(text, fake_quality) if text else transcribe(audio_path)
    with stage(timings, "decision"):
        decision = decide(tr, memory)
    with stage(timings, "reply_total"):
        twin_reply, plan = reply(decision, tr, memory)
    timings["tools_parallel"] = plan["timings"].get("tools_parallel", 0.0)
    timings["llm"] = plan["timings"].get("llm", 0.0)

    memory.add_heard(tr["text"])
    if twin_reply.action in ("answer", "defer", "ask_to_repeat") and twin_reply.reply_text:
        memory.add_said(twin_reply.reply_text)

    with stage(timings, "speech"):
        audio = speak(twin_reply, muted=muted)
    timings["total"] = round(time.perf_counter() - t_all, 3)

    log = config.OUT_DIR / "timings.csv"
    new = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if new:
            w.writerow(["turn", "action", *timings.keys()])
        w.writerow([_turn["n"], twin_reply.action, *timings.values()])

    return {"transcript": tr, "decision": decision, "tree": render_tree(decision),
            "tool_calls": list(CALL_LOG), "plan": plan, "reply": twin_reply.model_dump(),
            "audio": audio, "timings": timings}


Writing twin.py


## 2.13 `app.py` — Gradio demo
Record or upload a question, then see the transcript, the decision tree with pruned branches, the tool calls with JSON arguments, the final `TwinReply`, the stage timings, and play the audio. Includes a **kill switch** and a button to clear memory.

In [ ]:
%%writefile app.py
"""Gradio demo. Run:  python app.py   (in Colab: !python app.py or import and call demo.launch(share=True))"""
import gradio as gr

from twin import MEMORY, run_turn


def handle(audio, typed, muted):
    if not audio and not (typed or "").strip():
        return "Record, upload, or type a question first.", "", [], {}, {}, None, {}
    out = run_turn(audio_path=audio, text=(typed or "").strip() or None, muted=muted)
    tr = out["transcript"]
    shown = f"{tr['text']}\n\n[language: {tr['language']} | whisper: {tr['whisper_language']} | quality: {tr['quality']}]"
    return shown, out["tree"], out["tool_calls"], out["reply"], out["plan"], out["audio"], out["timings"]


def reset():
    MEMORY.reset()
    return "Memory cleared."


with gr.Blocks(title="Class Twin") as demo:
    gr.Markdown("## Class Twin - Phase 1\nRun only in meetings where the instructor knows the twin is on. "
                "Never for attendance, quizzes or exams.")
    with gr.Row():
        with gr.Column():
            audio = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Instructor question")
            typed = gr.Textbox(label="...or type it (skips speech-to-text, for debugging)")
            muted = gr.Checkbox(label="Kill switch: mute the twin (no audio)")
            go = gr.Button("Run", variant="primary")
            clear = gr.Button("Clear meeting memory")
            status = gr.Textbox(label="Status", interactive=False)
        with gr.Column():
            transcript = gr.Textbox(label="Transcript", lines=3)
            tree = gr.Textbox(label="Decision tree (with pruned branches)", lines=14)
    with gr.Row():
        calls = gr.JSON(label="Tool calls + JSON arguments")
        final = gr.JSON(label="Final TwinReply")
    with gr.Row():
        plan = gr.JSON(label="Dependency graph run (waves = parallel groups)")
        timings = gr.JSON(label="Stage timings (seconds)")
    out_audio = gr.Audio(label="Twin voice", type="filepath")
    go.click(handle, [audio, typed, muted], [transcript, tree, calls, final, plan, out_audio, timings])
    clear.click(reset, None, status)

if __name__ == "__main__":
    demo.launch(share=True)


Writing app.py


## 2.14 `run_tests.py` — test runner
Runs the 8 tests in order (memory is shared, so test 6 can use earlier answers) and writes a results table with expected vs. actual action.

In [ ]:
%%writefile run_tests.py
"""Run the 8 tests in order (memory is shared, so test 6 can use what the twin said earlier).
  python run_tests.py --mode text    # uses the "text" field (fast, no audio needed)
  python run_tests.py --mode audio   # uses the "audio" files in tests/audio/
Writes outputs/results.md and outputs/tests/test_N.json"""
import argparse
import json
from pathlib import Path

import config
from twin import MEMORY, run_turn

ap = argparse.ArgumentParser()
ap.add_argument("--mode", choices=["text", "audio"], default="text")
args = ap.parse_args()

tests = json.loads(Path(__file__).with_name("tests").joinpath("tests.json").read_text(encoding="utf-8"))
MEMORY.reset()
rows = []
for t in tests:
    if args.mode == "audio":
        out = run_turn(audio_path=str(Path(__file__).parent / t["audio"]))
    else:
        out = run_turn(text=t["text"], fake_quality=t.get("fake_quality"))
    exp = t["expected"] if isinstance(t["expected"], list) else [t["expected"]]
    actual = out["reply"]["action"]
    (config.OUT_DIR / "tests" / f"test_{t['id']}.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    rows.append((t["id"], t["name"], "/".join(exp), actual, "✅" if actual in exp else "❌",
                 out["reply"]["language"], out["transcript"]["quality"], out["timings"]["total"]))
    print(f"[{t['id']}] expected {exp} -> {actual}\n{out['tree']}\nreply: {out['reply']['reply_text']}\n")

md = ["| # | Test | Expected | Actual | Match | Lang | Quality | Total s |", "|---|---|---|---|---|---|---|---|"]
md += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
(config.OUT_DIR / "results.md").write_text("\n".join(md), encoding="utf-8")
print("\n".join(md))


Writing run_tests.py


## 2.15 `tests/tests.json` — the 8 test cases
Each test has the question text, the audio file name, and the expected action.

In [ ]:
%%writefile tests/tests.json
[
  {"id": 1, "name": "EN, addressed to me", "audio": "tests/audio/t1.m4a",
   "text": "Karim, what's the difference between RAG and fine-tuning?", "expected": "answer"},
  {"id": 2, "name": "Egyptian Arabic, addressed to me", "audio": "tests/audio/t2.m4a",
   "text": "يا كريم، إيه الفرق بين الـ RAG والـ fine-tuning؟", "expected": "answer"},
  {"id": 3, "name": "Mixed, addressed to me", "audio": "tests/audio/t3.m4a",
   "text": "يا كريم، ممكن تشرحلنا الـ pruning في الـ decision tree بيعمل إيه؟", "expected": "answer"},
  {"id": 4, "name": "Addressed to another student", "audio": "tests/audio/t4.m4a",
   "text": "Sara, can you share your screen?", "expected": "stay_silent"},
  {"id": 5, "name": "Open question to the class", "audio": "tests/audio/t5.m4a",
   "text": "Anyone? What does a CycleError mean?", "expected": ["answer", "stay_silent"]},
  {"id": 6, "name": "Follow-up needing memory", "audio": "tests/audio/t6.m4a",
   "text": "Karim, and how is that different from what you said earlier?", "expected": "answer"},
  {"id": 7, "name": "Outside my knowledge (Kubernetes)", "audio": "tests/audio/t7.m4a",
   "text": "Karim, how did your assignment use Kubernetes?", "expected": "defer"},
  {"id": 8, "name": "Unclear audio", "audio": "tests/audio/t8.m4a",
   "text": "uh the... can you", "fake_quality": 0.2, "expected": "ask_to_repeat"}
]


Overwriting tests/tests.json


---
# 🧪 Part 3 — Run and test
## 3.1 Run the 8 tests
Use `--mode text` to test with typed questions (no audio needed). When your recordings are ready (`tests/audio/t1.m4a` … `t8.m4a`), use `--mode audio`.

In [ ]:
!pip install -U edge-tts gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ClassTwin")
p = BASE / "speak.py"
s = p.read_text(encoding="utf-8")

old = '''    import edge_tts
    path = config.OUT_DIR / "audio" / f"reply_{int(time.time() * 1000)}.mp3"
    _run(edge_tts.Communicate(reply.reply_text, config.VOICES[reply.language]).save(str(path)))
    return str(path)'''

new = '''    path = config.OUT_DIR / "audio" / f"reply_{int(time.time() * 1000)}.mp3"
    try:
        import edge_tts
        _run(edge_tts.Communicate(reply.reply_text, config.VOICES[reply.language]).save(str(path)))
        return str(path)
    except Exception as e:
        print(f"[speak] edge-tts failed ({type(e).__name__}: {str(e)[:80]}), trying gTTS")
    try:
        from gtts import gTTS
        gTTS(reply.reply_text, lang="en" if reply.language == "en" else "ar").save(str(path))
        return str(path)
    except Exception as e:
        print(f"[speak] gTTS failed too ({type(e).__name__}), continuing without audio")
        return None'''

if "gTTS" in s:
    print("already patched")
elif old not in s:
    print("!! target text not found, nothing changed")
else:
    p.write_text(s.replace(old, new, 1), encoding="utf-8")
    print("speak.py patched")

speak.py patched


In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/ClassTwin/decide.py")
s = p.read_text(encoding="utf-8")

if "[:2000]" in s:
    print("already patched")
elif "h['text'][:350]" not in s:
    print("!! target text not found, nothing changed")
else:
    p.write_text(s.replace("h['text'][:350]", "h['text'][:2000]", 1), encoding="utf-8")
    print("decide.py patched")

decide.py patched


In [5]:
!python run_tests.py --mode text

Loading weights: 100% 199/199 [00:00<00:00, 5536.72it/s]
[1] expected ['answer'] -> answer
root: should I speak?   (threshold 0.6)
  name heard: karim | vocative: Karim | question: True | open: False | follow-up: False | quality: 0.9
├─ ask_to_repeat [check] score 0.00 (min 0.60)  ✂ pruned
│     why: quality 0.90 (min 0.45), 9 words
├─ stay_silent   [check+llm] score 0.00 (min 0.60)  ✂ pruned
│     why: no cheap reason to stay silent; LLM 0.00
├─ defer         [llm] score 0.00 (min 0.60)  ✂ pruned
│     why: LLM score
└─ answer        [llm] score 1.00 (min 0.60)  ★ SELECTED
      why: LLM score (threshold 0.60)
LLM reason: Direct question to Karim; definitions available in notes
reply: RAG retrieves relevant text from my documents and injects it into the prompt so the answer is grounded, while fine‑tuning changes the model’s weights to learn a new task or style. In AgriVision I built a RAG pipeline with FAISS, Sentence‑Transformers and Llama 3.1 because the disease info needs to stay u

In [6]:
   %cd /content/drive/MyDrive/ClassTwin
   !python grounding.py
   !python run_forced_tests.py

/content/drive/MyDrive/ClassTwin
OK    RAG retrieves relevant text from my documents and injects it into the  []
OK    RAG بيسترجع نصوص ذات صلة من الدوكومنتات وبيحطها في الـ prompt عشان الإ []
OK    في AgriVision استخدمت Random Forest متظبط (calibrated) عشان أرشح المحص []
OK    الـ pruning في الـ decision tree هو قص الفروع اللي مالهاش فرصة تكسب أو []
OK    Overfitting means the model memorizes the training data: high training []
OK    I haven't worked on that, so I don't want to guess. I can look into it []
CAUGHT البروننج هو قص الفروع اللي مالهاش فائدة. في مشروع AgriVision استخدمت d [['decision tree', 'pruning']]
CAUGHT في مشروع الـ DEPI استخدمنا decision tree لتحديد سياسات الرد للـ agent  [['validation accuracy', 'overfitting']]
CAUGHT I used Kubernetes in my assignment to deploy the model. [['kubernetes']]
CAUGHT In AgriVision we reached 98% accuracy with MobileNetV2. [['98']]

ALL GOOD
PASS  1. Normal ending
      reason=model_stopped iterations=2/4 tools=['search_my_notes', 'get_s

In [4]:
%cd /content/drive/MyDrive/ClassTwin
!python run_tests.py --mode audio

/content/drive/MyDrive/ClassTwin
modules.json: 100% 387/387 [00:00<00:00, 1.46MB/s]
README.md: 100% 498k/498k [00:00<00:00, 5.89MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 250kB/s]
config.json: 100% 655/655 [00:00<00:00, 1.95MB/s]

model.safetensors: downloading bytes:  49% 229M/471M [00:03<00:02, 80.5MB/s, 18.4MB/s  ]
model.safetensors: downloading bytes:  60% 283M/471M [00:03<00:01, 128MB/s, 21.7MB/s  ]
model.safetensors: downloading bytes:  66% 311M/471M [00:04<00:01, 131MB/s, 24.5MB/s  ]
model.safetensors: downloading bytes: 100% 315M/315M [00:04<00:00, 65.9MB/s, 25.6MB/s  ]
model.safetensors: reconstructing file: 100% 471M/471M [00:04<00:00, 98.6MB/s, 40.8MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 14507.36it/s]
tokenizer_config.json: 100% 443/443 [00:00<00:00, 1.59MB/s]

sentencepiece.bpe.model: downloading bytes:  47% 2.38M/5.07M [00:01<00:00, 3.06MB/s, 53.0kB/s  ]
sentencepiece.bpe.model: downloading bytes: 100% 3.62M/3.62M [00:01<00:00, 2.35MB/s,  346

## 3.2 Results table
Expected vs. actual action for each test. Paste it into your README.

In [ ]:
print(open(os.environ['TWIN_OUT'] + '/results.md', encoding='utf-8').read())

---
# 🖥️ Part 4 — Gradio demo
Launches the demo with a public link. Use it for your 2–3 minute screen recording.

In [ ]:
from app import demo
demo.launch(share=True, debug=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b5c7e2ec0505669fd5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b5c7e2ec0505669fd5.gradio.live


# Voice server on Colab

In [8]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ClassTwin
!pip install -q coqui-tts gradio soundfile

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ClassTwin
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.2/850.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.8 MB/s eta 0:00:00


In [13]:
import os
os.environ["TWIN_TTS_PASS"] = "karim"
os.environ["TWIN_VOICE_DIR"] = "/content/drive/MyDrive/ClassTwin/twin_data/voice"
%run xtts_server_colab.py

device: cpu


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

fast path: conditioning latents cached
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20a452ec8f5b3e29d6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
from gradio_client import Client
from IPython.display import Audio
c = Client("https://20a452ec8f5b3e29d6.gradio.live/", auth=("twin", "karim"))
Audio(c.predict("Hello, this is a test of my voice, I love all of you.", "en", api_name="/synthesize"))

Loaded as API: https://20a452ec8f5b3e29d6.gradio.live/


# VB-CABLE on Windows

In [20]:
!python audio_io.py --list

Traceback (most recent call last):
  File "/content/drive/MyDrive/ClassTwin/audio_io.py", line 216, in <module>
    list_devices()
    ~~~~~~~~~~~~^^
  File "/content/drive/MyDrive/ClassTwin/audio_io.py", line 46, in list_devices
    import sounddevice as sd
ModuleNotFoundError: No module named 'sounddevice'


---
# 📝 Part 5 — Still to do (README)
- [ ] Paste the results table
- [ ] Write **one thing the twin got wrong, and why**
- [ ] Blind test: read 3 twin replies + 3 of your real answers to a classmate and report how many they identified
- [ ] Keep `voice/` out of the public repo

In [ ]:
import subprocess, pathlib
V = pathlib.Path("/content/drive/MyDrive/ClassTwin/twin_data/voice")
for f in V.glob("*"):
    if f.suffix.lower() in {".m4a", ".mp3", ".aac", ".ogg", ".opus", ".mp4", ".amr", ".3gp"}:
        out = f.with_suffix(".wav")
        subprocess.run(["ffmpeg", "-y", "-i", str(f), "-ac", "1", "-ar", "24000", "-sample_fmt", "s16", str(out)], check=True)
        print("converted", f.name, "->", out.name)

converted consent.mp4 -> consent.wav
converted arabic_sample.mp4 -> arabic_sample.wav
converted english_sample.mp4 -> english_sample.wav


In [ ]:
import subprocess, pathlib
V = pathlib.Path("/content/drive/MyDrive/ClassTwin/tests/blind")
for f in V.glob("*"):
    if f.suffix.lower() in {".m4a", ".mp3", ".aac", ".ogg", ".opus", ".mp4", ".amr", ".3gp"}:
        out = f.with_suffix(".wav")
        subprocess.run(["ffmpeg", "-y", "-i", str(f), "-ac", "1", "-ar", "24000", "-sample_fmt", "s16", str(out)], check=True)
        print("converted", f.name, "->", out.name)

converted real_1.mp4 -> real_1.wav
converted real_2.mp4 -> real_2.wav
converted real_3.mp4 -> real_3.wav
